In [1]:
#Packages Import
import json
import os
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [2]:
#Validation Script
def run_master_validation(root_folder):
    stats = []
    
    #Traversing Ticker/Year structure
    for root, dirs, files in os.walk(root_folder):
        for file in files:
            if file.lower().endswith('.json'):
                try:
                    with open(os.path.join(root, file), 'r', encoding='utf-8') as f:
                        data = json.load(f)
                    
                    #Extracts identifying info
                    #Using .get() with defaults to prevent crashes if a file is malformed
                    meta = data.get('metadata', {})
                    ticker = meta.get('ticker', 'UNKNOWN')
                    year = meta.get('fiscal_year', 'N/A')
                    
                    transcript = data.get('transcript', {})
                    full_text = transcript.get('full_transcript_text', "")
                    sections = transcript.get('sections', [])
                    
                    exec_turns = 0
                    analyst_turns = 0
                    segmented_word_count = 0
                    
                    #Audit turns and words within a specific file
                    for section in sections:
                        content = section.get('content', [])
                        for turn in content:
                            text = turn.get('text', '')
                            role = str(turn.get('role', '')).lower()
                            
                            segmented_word_count += len(text.split())
                            if role == 'executive':
                                exec_turns += 1
                            elif role == 'analyst':
                                analyst_turns += 1

                    full_word_count = len(full_text.split())
                    
                    stats.append({
                        "Ticker": ticker,
                        "Year": year,
                        "Exec_Turns": exec_turns,
                        "Analyst_Turns": analyst_turns,
                        "Full_Words": full_word_count,
                        "Seg_Words": segmented_word_count,
                        "Variance": full_word_count - segmented_word_count
                    })
                except Exception as e:
                    print(f"Error in file {file}: {e}")
                    continue

    if not stats:
        print("No data collected. Check file paths.")
        return None

    #Converting to DataFrame
    df = pd.DataFrame(stats)
    
    #Grouping by Ticker for the final 15-ticker view
    master_report = df.groupby('Ticker').agg({
        'Exec_Turns': 'sum',
        'Analyst_Turns': 'sum',
        'Full_Words': 'sum',
        'Seg_Words': 'sum',
        'Variance': 'sum'
    }).reset_index() 

    #Calculating Analyst Ratio
    master_report['Analyst_Ratio'] = (master_report['Analyst_Turns'] / (master_report['Exec_Turns'] + 0.0001)).round(2)
    
    #Status Logic
    def get_status(row):
        if row['Variance'] != 0: return "LOSS"
        if row['Analyst_Ratio'] < 0.2: return "ANOMALY"
        return "OK"

    master_report['Status'] = master_report.apply(get_status, axis=1)
    
    return master_report

In [3]:
#Execution
output_path = r'C:\Users\USER\PycharmProjects\ThesisProject\Data\Transcripts\Normalized'
df_final_report = run_master_validation(output_path)
if df_final_report is not None:
  print(df_final_report.to_string(index=False))

Ticker  Exec_Turns  Analyst_Turns  Full_Words  Seg_Words  Variance  Analyst_Ratio Status
  ADSK         614            560      192454     192454         0           0.91     OK
   APA         705            650      165551     165551         0           0.92     OK
  APTV         679            529      183750     183750         0           0.78     OK
   BKR         500            399      181351     181351         0           0.80     OK
    BX         611            321      176427     176427         0           0.53     OK
 CMCSA         414            331      187898     187898         0           0.80     OK
   DAL        1062            950      206449     206449         0           0.89     OK
   FOX         340            122      113502     113502         0           0.36     OK
 GOOGL         403            253      145712     145712         0           0.63     OK
    GS         620            527      216664     216664         0           0.85     OK
   HPE         495   